In [1]:
import pandas as pd
import numpy as np

pharm_data = pd.read_csv("C:\\Users\\USER\\Downloads\\bloom_nigeria_pharma_cleaned_v2.csv")
print(pharm_data.shape)

(2000, 38)


In [2]:
print("STOCKOUT & SUPPLY RISK ANALYSIS\n")

# 1. Overall status distribution
print("--- Status Distribution ---")
print(pharm_data['Status'].value_counts())
print(f"\nTotal records: {len(pharm_data)}")
print(f"At-risk records (Critical + Out_of_Stock + Low_Stock): {pharm_data['Status'].isin(['Critical', 'Out_of_Stock', 'Low_Stock']).sum()}")

STOCKOUT & SUPPLY RISK ANALYSIS

--- Status Distribution ---
Status
Expired                   847
In_Stock                  531
Out_of_Stock              253
Low_Stock                 119
Critical                  109
Expiring_Within_30Days     94
Counterfeit_Risk           47
Name: count, dtype: int64

Total records: 2000
At-risk records (Critical + Out_of_Stock + Low_Stock): 481


In [3]:
# 2. Stockout rate by state
print("--- Stockout Rate by State ---")
stockout_by_state = pharm_data[pharm_data['Status'] == 'Out_of_Stock'].groupby('State').size().reset_index(name='Stockout_Count')
total_by_state = pharm_data.groupby('State').size().reset_index(name='Total_Count')

state_analysis = stockout_by_state.merge(total_by_state, on='State')
state_analysis['Stockout_Rate_%'] = (state_analysis['Stockout_Count'] / state_analysis['Total_Count'] * 100).round(1)
state_analysis = state_analysis.sort_values('Stockout_Rate_%', ascending=False)
print(state_analysis.head(10))

--- Stockout Rate by State ---
       State  Stockout_Count  Total_Count  Stockout_Rate_%
20   Katsina              23           67             34.3
32    Sokoto              24           71             33.8
19      Kano              15           63             23.8
34   Zamfara              14           61             23.0
24  Nasarawa               9           44             20.5
33      Yobe              13           65             20.0
7      Borno              12           60             20.0
17    Jigawa              12           60             20.0
25     Niger              11           62             17.7
6      Benue               9           54             16.7


In [5]:
# 3. Most stocked out medicines
print("Most Stocked Out Medicines")
stockout_by_medicine = pharm_data[pharm_data['Status'] == 'Out_of_Stock'].groupby('Medicine_Name').size().reset_index(name='Stockout_Count')
stockout_by_medicine = stockout_by_medicine.sort_values('Stockout_Count', ascending=False)
print(stockout_by_medicine.head(10))

Most Stocked Out Medicines
                        Medicine_Name  Stockout_Count
17         Morphine 10mg/Ml Injection              14
19           Opv (Oral Polio Vaccine)              13
18                   Nevirapine 200mg              13
24               Yellow Fever Vaccine              13
6            Ceftriaxone 1G Injection              13
7         Chloroquine Phosphate 250mg              12
4                  Azithromycin 500mg              12
22  Tenofovir/Lamivudine/Dolutegravir              12
0          Amoxicillin 250mg Capsules              11
2    Artemether-Lumefantrine 20/120mg              11


In [6]:
# 4. Average stockout events by facility type
print("--- Average Stockout Events by Facility Type ---")
stockout_by_facility = pharm_data.groupby('Facility_Type')['Stockout_Events_Last_6Months'].mean().round(2).reset_index()
stockout_by_facility = stockout_by_facility.sort_values('Stockout_Events_Last_6Months', ascending=False)
print(stockout_by_facility)

--- Average Stockout Events by Facility Type ---
       Facility_Type  Stockout_Events_Last_6Months
1                PHC                          3.15
0   General Hospital                          2.76
2  Teaching Hospital                          2.40


In [7]:
# 5. High risk facilities - multiple stockout events
print("--- Facilities with Most Stockout Events ---")
high_risk = pharm_data.groupby(['Facility_Name', 'State', 'Facility_Type'])['Stockout_Events_Last_6Months'].mean().round(1).reset_index()
high_risk = high_risk.sort_values('Stockout_Events_Last_6Months', ascending=False)
print(high_risk.head(10))

--- Facilities with Most Stockout Events ---
                           Facility_Name    State     Facility_Type  \
369             General Hospital Rural A    Gombe  General Hospital   
478               Kaga District Hospital    Borno  General Hospital   
485   Kaura Namoda Primary Health Centre  Zamfara               PHC   
52             Central District Hospital    Benue  General Hospital   
1116                            West PHC    Benue               PHC   
423              General Hospital Ungogo     Yobe               PHC   
849                          Ward 10 PHC    Borno               PHC   
809                    Talata Mafara PHC  Zamfara               PHC   
812       Tambuwal Primary Health Centre   Sokoto               PHC   
762          South Primary Health Centre  Plateau               PHC   

      Stockout_Events_Last_6Months  
369                            8.0  
478                            8.0  
485                            8.0  
52                       

In [8]:
print("=== COLD CHAIN & VACCINE INTEGRITY ANALYSIS ===\n")

# 1. Overall cold chain compliance
print("--- Overall Cold Chain Compliance ---")
print(pharm_data['Cold_Chain_Compliant'].value_counts())
total = len(pharm_data)
compliant = (pharm_data['Cold_Chain_Compliant'] == 'Yes').sum()
print(f"\nCompliance Rate: {round(compliant/total*100, 1)}%")

=== COLD CHAIN & VACCINE INTEGRITY ANALYSIS ===

--- Overall Cold Chain Compliance ---
Cold_Chain_Compliant
Yes    1770
No      230
Name: count, dtype: int64

Compliance Rate: 88.5%


In [9]:
# 2. Cold chain compliance by facility type
print("--- Cold Chain Compliance by Facility Type ---")
print(pharm_data.groupby('Facility_Type')['Cold_Chain_Compliant'].value_counts())

--- Cold Chain Compliance by Facility Type ---
Facility_Type      Cold_Chain_Compliant
General Hospital   Yes                      536
                   No                        67
PHC                Yes                     1083
                   No                       146
Teaching Hospital  Yes                      151
                   No                        17
Name: count, dtype: int64


In [10]:
# Calculate failure rate by facility type
print("--- Cold Chain Failure Rate by Facility Type ---")
compliance = pharm_data.groupby('Facility_Type')['Cold_Chain_Compliant'].value_counts(normalize=True).mul(100).round(1).reset_index(name='Percentage')
print(compliance[compliance['Cold_Chain_Compliant'] == 'No'])

--- Cold Chain Failure Rate by Facility Type ---
       Facility_Type Cold_Chain_Compliant  Percentage
1   General Hospital                   No        11.1
3                PHC                   No        11.9
5  Teaching Hospital                   No        10.1


In [12]:
# 3. Cold chain compliance by state
print("--- Cold Chain Failure Rate by State ---")
state_compliance = pharm_data.groupby('State')['Cold_Chain_Compliant'].apply(
    lambda x: (x == 'No').sum() / len(x) * 100
).round(1).reset_index(name='Failure_Rate_%')
state_compliance = state_compliance.sort_values('Failure_Rate_%', ascending=False)
print(state_compliance.head(10))

--- Cold Chain Failure Rate by State ---
       State  Failure_Rate_%
1    Adamawa            25.0
22     Kwara            19.3
29       Oyo            18.5
18    Kaduna            17.3
7      Borno            16.7
20   Katsina            16.4
34   Zamfara            14.8
9      Delta            14.3
16       Imo            14.0
24  Nasarawa            13.6


In [13]:
# 4. Link between cold chain failure and wastage
print("--- Average Wastage by Cold Chain Compliance ---")
wastage = pharm_data.groupby('Cold_Chain_Compliant')['Wastage_Percentage_Last_Quarter'].mean().round(2)
print(wastage)

--- Average Wastage by Cold Chain Compliance ---
Cold_Chain_Compliant
No     34.88
Yes     6.77
Name: Wastage_Percentage_Last_Quarter, dtype: float64


In [14]:
# 5. Vaccine specific analysis
print("--- Vaccine Wastage vs Non-Vaccine ---")
pharm_data['Is_Vaccine'] = pharm_data['Medicine_Category'] == 'Vaccine'
print(pharm_data.groupby('Is_Vaccine')['Wastage_Percentage_Last_Quarter'].mean().round(2))

--- Vaccine Wastage vs Non-Vaccine ---
Is_Vaccine
False     6.84
True     20.65
Name: Wastage_Percentage_Last_Quarter, dtype: float64


In [15]:
# 6. Cold chain compliance by geopolitical zone
print("--- Cold Chain Failure Rate by Geopolitical Zone ---")
zone_compliance = pharm_data.groupby('Geopolitical_Zone')['Cold_Chain_Compliant'].apply(
    lambda x: (x == 'No').sum() / len(x) * 100
).round(1).reset_index(name='Failure_Rate_%')
zone_compliance = zone_compliance.sort_values('Failure_Rate_%', ascending=False)
print(zone_compliance)

--- Cold Chain Failure Rate by Geopolitical Zone ---
  Geopolitical_Zone  Failure_Rate_%
1        North_East            15.2
2        North_West            13.1
0     North_Central            10.3
3        South_East            10.2
4       South_South            10.1
5        South_West             9.8


In [16]:
print("=== REGIONAL DISPARITIES ANALYSIS ===\n")

# 1. Stock status by geopolitical zone
print("--- Stock Status by Geopolitical Zone ---")
zone_status = pharm_data.groupby(['Geopolitical_Zone', 'Status']).size().unstack(fill_value=0)
print(zone_status)

=== REGIONAL DISPARITIES ANALYSIS ===

--- Stock Status by Geopolitical Zone ---
Status             Counterfeit_Risk  Critical  Expired  \
Geopolitical_Zone                                        
North_Central                     8        18      144   
North_East                        6        30      125   
North_West                        7        33      176   
South_East                       13        10      137   
South_South                       6        12      125   
South_West                        7         6      140   

Status             Expiring_Within_30Days  In_Stock  Low_Stock  Out_of_Stock  
Geopolitical_Zone                                                             
North_Central                          12        78         22            47  
North_East                             15        50         22            41  
North_West                             17        51         53            98  
South_East                             17        93        

In [17]:
# 2. At-risk rate by zone
print("--- At-Risk Rate by Geopolitical Zone ---")
zone_risk = pharm_data.groupby('Geopolitical_Zone').apply(
    lambda x: (x['Status'].isin(['Critical', 'Out_of_Stock', 'Low_Stock'])).sum() / len(x) * 100
).round(1).reset_index(name='At_Risk_%')
zone_risk = zone_risk.sort_values('At_Risk_%', ascending=False)
print(zone_risk)

--- At-Risk Rate by Geopolitical Zone ---
  Geopolitical_Zone  At_Risk_%
2        North_West       42.3
1        North_East       32.2
0     North_Central       26.4
3        South_East       14.5
4       South_South       12.9
5        South_West        9.8


In [18]:
# 3. Average days of stock remaining by zone
print("--- Average Days of Stock Remaining by Zone ---")
days_stock = pharm_data.groupby('Geopolitical_Zone')['Days_Of_Stock_Remaining'].mean().round(1).reset_index()
days_stock = days_stock.sort_values('Days_Of_Stock_Remaining')
print(days_stock)

--- Average Days of Stock Remaining by Zone ---
  Geopolitical_Zone  Days_Of_Stock_Remaining
2        North_West                     27.5
1        North_East                     36.2
0     North_Central                     77.4
4       South_South                    101.7
3        South_East                    110.4
5        South_West                    114.7


In [19]:
# 4. Average stock value by zone
print("--- Average Stock Value by Zone (Naira) ---")
stock_value = pharm_data.groupby('Geopolitical_Zone')['Total_Stock_Value'].mean().round(0).reset_index()
stock_value = stock_value.sort_values('Total_Stock_Value')
print(stock_value)

--- Average Stock Value by Zone (Naira) ---
  Geopolitical_Zone  Total_Stock_Value
2        North_West           704405.0
1        North_East          1695577.0
0     North_Central          2647416.0
4       South_South          2746619.0
3        South_East          3065269.0
5        South_West          3114358.0


In [20]:
# 5. Wastage by zone
print("--- Average Wastage by Geopolitical Zone ---")
wastage_zone = pharm_data.groupby('Geopolitical_Zone')['Wastage_Percentage_Last_Quarter'].mean().round(2).reset_index()
wastage_zone = wastage_zone.sort_values('Wastage_Percentage_Last_Quarter', ascending=False)
print(wastage_zone)

--- Average Wastage by Geopolitical Zone ---
  Geopolitical_Zone  Wastage_Percentage_Last_Quarter
1        North_East                            10.85
2        North_West                            10.48
4       South_South                            10.00
5        South_West                             9.75
3        South_East                             9.61
0     North_Central                             9.27


In [21]:
print("=== TRACK & TRACE / COUNTERFEIT RISK ANALYSIS ===\n")

# 1. Track & Trace adoption
print("--- Track & Trace Enabled ---")
print(pharm_data['Track_Trace_Enabled'].value_counts())
total = len(pharm_data)
enabled = (pharm_data['Track_Trace_Enabled'] == 'Yes').sum()
print(f"\nAdoption Rate: {round(enabled/total*100, 1)}%")

=== TRACK & TRACE / COUNTERFEIT RISK ANALYSIS ===

--- Track & Trace Enabled ---
Track_Trace_Enabled
No     1650
Yes     350
Name: count, dtype: int64

Adoption Rate: 17.5%


In [22]:
# 2. Serialization code coverage
print("--- Serialization Code Coverage ---")
serialized = pharm_data['Serialization_Code'].notna().sum()
print(f"Medicines with Serialization Code: {serialized}")
print(f"Medicines without Serialization Code: {total - serialized}")
print(f"Serialization Coverage: {round(serialized/total*100, 1)}%")

--- Serialization Code Coverage ---
Medicines with Serialization Code: 350
Medicines without Serialization Code: 1650
Serialization Coverage: 17.5%


In [23]:
# 3. Counterfeit risk analysis
print("--- Counterfeit Risk by Zone ---")
counterfeit = pharm_data[pharm_data['Status'] == 'Counterfeit_Risk']
print(f"Total Counterfeit Risk records: {len(counterfeit)}")
print(f"Percentage of total: {round(len(counterfeit)/total*100, 1)}%\n")

print(counterfeit.groupby('Geopolitical_Zone').size().reset_index(name='Count').sort_values('Count', ascending=False))

--- Counterfeit Risk by Zone ---
Total Counterfeit Risk records: 47
Percentage of total: 2.4%

  Geopolitical_Zone  Count
3        South_East     13
0     North_Central      8
5        South_West      7
2        North_West      7
1        North_East      6
4       South_South      6


In [25]:
# 4. Counterfeit risk by medicine
print("--- Counterfeit Risk by Medicine ---")
print(counterfeit['Medicine_Name'].value_counts())

--- Counterfeit Risk by Medicine ---
Medicine_Name
Tenofovir/Lamivudine/Dolutegravir    6
Nevirapine 200mg                     5
Morphine 10mg/Ml Injection           4
Artemether-Lumefantrine 20/120mg     4
Measles-Rubella Vaccine              3
Ceftriaxone 1G Injection             3
Metronidazole 400mg                  2
Artesunate Injection 60mg            2
Hpv Vaccine                          2
Amoxicillin-Clavulanate 625mg        2
Azithromycin 500mg                   2
Lopinavir/Ritonavir 200/50mg         2
Opv (Oral Polio Vaccine)             2
Amoxicillin 250mg Capsules           1
Yellow Fever Vaccine                 1
Ibuprofen 400mg                      1
Paracetamol 500mg                    1
Tramadol 50mg                        1
Bcg Vaccine                          1
Dihydroartemisinin-Piperaquine       1
Efavirenz/Lamivudine/Tenofovir       1
Name: count, dtype: int64


In [26]:
# 5. Does Track & Trace reduce counterfeit risk?
print("--- Counterfeit Risk vs Track & Trace ---")
tt_counterfeit = pharm_data.groupby('Track_Trace_Enabled').apply(
    lambda x: (x['Status'] == 'Counterfeit_Risk').sum() / len(x) * 100
).round(2).reset_index(name='Counterfeit_Rate_%')
print(tt_counterfeit)

--- Counterfeit Risk vs Track & Trace ---
  Track_Trace_Enabled  Counterfeit_Rate_%
0                  No                2.85
1                 Yes                0.00


In [ ]:
# 6. Supply chain visibility coverage
print("--- Supply Chain Visibility Coverage ---")
visibility = pharm_data['Supply_Chain_Visibility'].value_counts(dropna=False)
print(visibility)
print(f"\nMissing/No Visibility: {pharm_data['Supply_Chain_Visibility'].isna().sum()} records")

--- Supply Chain Visibility Coverage ---
Supply_Chain_Visibility
NaN        1155
Partial     584
Full        261
Name: count, dtype: int64

Missing/No Visibility: 1155 records
